# Undersampling vs oversampling klasy `unknown`

Notebook sprawdza wpływ fizycznego resamplowania zbioru treningowego na klasyfikację komend głosowych. Eksperyment porównuje model trenowany na naturalnym rozkładzie danych, undersampling klasy `unknown` do liczności największej pozostałej klasy oraz wariant, który najpierw zmniejsza największą klasę do maksymalnie dwukrotności drugiej największej klasy, a następnie oversampluje wszystkie klasy do tego rozmiaru.

Walidacja i test zostają bez resamplowania. Dzięki temu metryki mierzą zachowanie modeli na naturalnym rozkładzie danych, a zmieniamy wyłącznie dane widziane w trakcie uczenia.

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from scripts import (
    DataFixedParams,
    DataGridParams,
    Experiment,
    FeatureFixedParams,
    FitFixedParams,
    FitGridParams,
    LABEL_ORDER,
    ModelGridParams,
    UNKNOWN_LABEL,
    experiment_grid_dataframe,
    expand_experiment_grid,
    prepare_experiment_data_files,
    train_experiment,
)
from scripts.data import PreparedDataFiles
from scripts.runner import data_key

LABEL_ORDER

('yes',
 'no',
 'up',
 'down',
 'left',
 'right',
 'on',
 'off',
 'stop',
 'go',
 'unknown',
 'silence')

## Konfiguracja

`unknown_fraction` jest ustawione na `0.06`, żeby klasa `unknown` nadal była większa od pojedynczych klas docelowych, ale żeby wariant oversamplingu nie tworzył niepraktycznie dużego zbioru treningowego. Konfiguracja używa obu architektur, małych modeli i `device="cuda"`, więc jest ustawiona pod krótkie uruchomienie na GPU.

In [2]:
OUTPUT_DIR = "reports/05_sampling_experiment"
ARCHITECTURES = ["lstm", "transformer"]
SELECTED_STRATEGIES = [
    "natural",
    "unknown_undersampled",
    "undersampled_then_oversampled",
]

DATA_GRID_CONFIG = {
    "train_fraction": 1,
    "validation_fraction": 1,
    "test_fraction": 1,
    "unknown_fraction": 1,
    "silence_samples": 2000,
    "sampling_strategy": "natural",
    "seed": 42,
}

FEATURE_CONFIG = {
    "n_mels": 64,
    "n_fft": 512,
    "hop_length": 160,
    "normalize": True,
}

FIT_FIXED_CONFIG = {
    "device": "cuda",
    "use_tqdm": True,
    "progress_backend": "terminal",
    "verbose": True,
    "early_stopping": True,
    "early_stopping_patience": 5,
    "early_stopping_min_delta": 0.001,
}

FIT_GRID_CONFIG = {
    "epochs": 5,
    "batch_size": 128,
    "learning_rate": 1e-3,
    "weight_decay": 1e-4,
}

MODEL_CONFIG = {
    "dropout": 0.2,
    "lstm_hidden_size": 64,
    "lstm_layers": 2,
    "lstm_bidirectional": True,
    "transformer_d_model": 64,
    "transformer_heads": 4,
    "transformer_layers": 2,
    "transformer_ff_dim": 128,
}

base_data_fixed = DataFixedParams(
    cache_dir=".cache/sampling_experiment_audio",
    reuse_cached_dataset=False,
    output_dir=OUTPUT_DIR,
)
base_data_grid = DataGridParams(**DATA_GRID_CONFIG)
base_feature_fixed = FeatureFixedParams(**FEATURE_CONFIG)
base_fit_fixed = FitFixedParams(**FIT_FIXED_CONFIG)
base_fit_grid = FitGridParams(**FIT_GRID_CONFIG)

pd.DataFrame(
    [
        {"group": "data", **DATA_GRID_CONFIG},
        {"group": "features", **FEATURE_CONFIG},
        {"group": "fit", **FIT_FIXED_CONFIG, **FIT_GRID_CONFIG},
        {"group": "model", **MODEL_CONFIG},
    ]
)

,group,train_fraction,validation_fraction,test_fraction,unknown_fraction,silence_samples,sampling_strategy,seed,n_mels,n_fft,...,learning_rate,weight_decay,dropout,lstm_hidden_size,lstm_layers,lstm_bidirectional,transformer_d_model,transformer_heads,transformer_layers,transformer_ff_dim
0,data,1.0,1.0,1.0,1.0,2000.0,natural,42.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,features,NaN,NaN,NaN,NaN,NaN,NaN,NaN,64.0,512.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,fit,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.001,0.0001,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,model,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,0.2,64.0,2.0,True,64.0,4.0,2.0,128.0


## Strategie resamplowania

Strategia `natural` zostawia zbiór treningowy bez undersamplingu i oversamplingu, żeby dać punkt odniesienia dla pozostałych wariantów.

Strategia `unknown_undersampled` losowo zmniejsza tylko klasę `unknown` do liczności największej klasy nie-`unknown`.

Strategia `undersampled_then_oversampled` najpierw sprawdza dwie największe klasy w naturalnym zbiorze treningowym. Jeśli największa klasa jest większa niż dwukrotność drugiej największej, zostaje losowo zmniejszona do tego limitu. Następnie wszystkie klasy są doprowadzane do tego docelowego rozmiaru przez oversampling mniejszych klas.

In [3]:
def class_counts(manifest: pd.DataFrame) -> pd.Series:
    return manifest["label"].value_counts().reindex(LABEL_ORDER, fill_value=0)


def largest_non_unknown_count(train_manifest: pd.DataFrame) -> int:
    counts = train_manifest[train_manifest["label"] != UNKNOWN_LABEL]["label"].value_counts()
    return int(counts.max())


def undersample_unknown(train_manifest: pd.DataFrame, seed: int) -> pd.DataFrame:
    target_count = largest_non_unknown_count(train_manifest)
    sampled_groups = []

    for label, group in train_manifest.groupby("label", sort=False):
        if label == UNKNOWN_LABEL and len(group) > target_count:
            sampled_groups.append(group.sample(n=target_count, random_state=seed))
        else:
            sampled_groups.append(group)

    return pd.concat(sampled_groups, ignore_index=True)


def capped_largest_target_count(train_manifest: pd.DataFrame) -> int:
    counts = train_manifest["label"].value_counts().sort_values(ascending=False)
    if counts.empty:
        raise ValueError("Cannot resample an empty train manifest")
    if len(counts) == 1:
        return int(counts.iloc[0])

    largest_count = int(counts.iloc[0])
    second_largest_count = int(counts.iloc[1])
    return min(largest_count, 2 * second_largest_count)


def oversample_to_count(train_manifest: pd.DataFrame, target_count: int, seed: int) -> pd.DataFrame:
    sampled_groups = []

    for label, group in train_manifest.groupby("label", sort=False):
        if len(group) >= target_count:
            sampled_groups.append(group.sample(n=target_count, random_state=seed) if len(group) > target_count else group)
            continue

        repetitions = target_count // len(group)
        remainder = target_count % len(group)
        parts = [group.copy() for _ in range(repetitions)]
        if remainder:
            parts.append(group.sample(n=remainder, replace=False, random_state=seed))
        sampled_groups.append(pd.concat(parts, ignore_index=True))

    resampled = pd.concat(sampled_groups, ignore_index=True)
    resampled["resample_copy_id"] = resampled.groupby(["label", "archive_path"]).cumcount()
    return resampled


def undersample_largest_then_oversample(train_manifest: pd.DataFrame, seed: int) -> pd.DataFrame:
    target_count = capped_largest_target_count(train_manifest)
    return oversample_to_count(train_manifest, target_count, seed)


def resample_train_manifest(train_manifest: pd.DataFrame, strategy: str, seed: int) -> pd.DataFrame:
    train_manifest = train_manifest.reset_index(drop=True).copy()

    if strategy == "natural":
        result = train_manifest
    elif strategy == "unknown_undersampled":
        result = undersample_unknown(train_manifest, seed)
    elif strategy == "undersampled_then_oversampled":
        result = undersample_largest_then_oversample(train_manifest, seed)
    else:
        raise ValueError(f"Unknown strategy: {strategy}")

    result = result.reset_index(drop=True).copy()
    result["balancing_strategy"] = strategy
    return result


def replace_train_manifest(prepared_files: PreparedDataFiles, train_manifest: pd.DataFrame) -> PreparedDataFiles:
    validation_manifest = prepared_files.validation_manifest.copy()
    test_manifest = prepared_files.test_manifest.copy()
    manifest = pd.concat([train_manifest, validation_manifest, test_manifest], ignore_index=True)

    return PreparedDataFiles(
        manifest=manifest,
        train_manifest=train_manifest,
        validation_manifest=validation_manifest,
        test_manifest=test_manifest,
        local_paths=prepared_files.local_paths,
    )

## Definicje eksperymentów

Każda strategia resamplowania jest trenowana osobno dla LSTM i Transformera. Architektury są takie same jak w poprzednich eksperymentach; zmienia się tylko rozkład danych treningowych.

In [4]:
def model_params_for(architecture: str) -> dict:
    return {
        "model_type": architecture,
        **MODEL_CONFIG,
    }


def make_experiment(strategy: str, architecture: str) -> Experiment:
    safe_strategy = strategy.replace("_", "-")
    return Experiment(
        name=f"05_sampling_{safe_strategy}_{architecture}",
        data_fixed=base_data_fixed,
        data_grid=base_data_grid,
        feature_fixed=base_feature_fixed,
        model_grid=ModelGridParams(**model_params_for(architecture)),
        fit_fixed=base_fit_fixed,
        fit_grid=base_fit_grid,
    )


experiments = {
    (strategy, architecture): make_experiment(strategy, architecture)
    for strategy in SELECTED_STRATEGIES
    for architecture in ARCHITECTURES
}

experiment_grid_dataframe(experiments[(SELECTED_STRATEGIES[0], ARCHITECTURES[0])])

,experiment,data.train_fraction,data.validation_fraction,data.test_fraction,data.unknown_fraction,data.silence_samples,data.sampling_strategy,data.seed,model.model_type,model.dropout,...,model.lstm_layers,model.lstm_bidirectional,model.transformer_d_model,model.transformer_heads,model.transformer_layers,model.transformer_ff_dim,fit.epochs,fit.batch_size,fit.learning_rate,fit.weight_decay
0,05_sampling_natural_lstm,1,1,1,1,2000,natural,42,lstm,0.2,...,2,True,64,4,2,128,10,128,0.001,0.0001


## Przygotowanie danych

Najpierw przygotowywany jest jeden manifest bazowy. Następnie tworzymy jego warianty z różnymi rozkładami klas w treningu. Pliki audio pozostają te same, więc cache ekstrakcji archiwum jest współdzielony.

In [5]:
data_cache_experiment = make_experiment("data_cache", ARCHITECTURES[0])
base_prepared_files = prepare_experiment_data_files(data_cache_experiment, DATA_GRID_CONFIG)

base_distribution = pd.crosstab(base_prepared_files.manifest["label"], base_prepared_files.manifest["split"])
base_distribution.reindex(LABEL_ORDER, fill_value=0)

Extracting archive: 100%|██████████| 1/1 [00:56<00:00, 56.40s/it]


split,test,train,validation
label,,,
yes,256,1860,261
no,252,1853,270
up,272,1843,260
down,253,1842,264
left,267,1839,247
right,259,1852,256
on,246,1864,257
off,262,1839,256
stop,249,1885,246


In [6]:
strategy_prepared_files = {}
strategy_distributions = []

for strategy in SELECTED_STRATEGIES:
    train_manifest = resample_train_manifest(
        base_prepared_files.train_manifest,
        strategy=strategy,
        seed=DATA_GRID_CONFIG["seed"],
    )
    strategy_prepared_files[strategy] = replace_train_manifest(base_prepared_files, train_manifest)

    counts = class_counts(train_manifest)
    for label, count in counts.items():
        strategy_distributions.append(
            {
                "strategy": strategy,
                "label": label,
                "train_count": int(count),
            }
        )

strategy_distribution = pd.DataFrame(strategy_distributions)
strategy_distribution.pivot(index="label", columns="strategy", values="train_count").reindex(LABEL_ORDER)

strategy,natural,undersampled_then_oversampled,unknown_undersampled
label,,,
yes,1860,3770,1860
no,1853,3770,1853
up,1843,3770,1843
down,1842,3770,1842
left,1839,3770,1839
right,1852,3770,1852
on,1864,3770,1864
off,1839,3770,1839
stop,1885,3770,1885


## Uruchomienie eksperymentów

Ta komórka uruchamia wszystkie strategie z `SELECTED_STRATEGIES` dla obu architektur. Wyniki każdego treningu są zapisywane w osobnym katalogu, a wspólne podsumowanie trafia do `reports/05_sampling_experiment/sampling_experiment_summary.csv`.

In [7]:
all_results = []

for strategy in SELECTED_STRATEGIES:
    prepared_files = strategy_prepared_files[strategy]

    for architecture in ARCHITECTURES:
        experiment = experiments[(strategy, architecture)]
        run = expand_experiment_grid(experiment)[0]
        prepared_by_key = {data_key(run): prepared_files}

        print(f"[{strategy} | {architecture}]")
        summary = train_experiment(experiment, prepared_by_key).copy()
        summary.insert(0, "strategy", strategy)
        summary.insert(1, "architecture", architecture)
        summary.insert(2, "train_examples", len(prepared_files.train_manifest))
        summary.insert(3, "unknown_train_examples", int((prepared_files.train_manifest["label"] == UNKNOWN_LABEL).sum()))
        all_results.append(summary)

all_results = pd.concat(all_results, ignore_index=True)
summary_path = Path(OUTPUT_DIR) / "sampling_experiment_summary.csv"
summary_path.parent.mkdir(parents=True, exist_ok=True)
all_results.to_csv(summary_path, index=False)
all_results.sort_values("test_accuracy", ascending=False)

[natural | lstm]
Starting experiment: 05_sampling_natural_lstm

Configuration run 1/1:
DATA (variable):
  - train_fraction: 1
  - validation_fraction: 1
  - test_fraction: 1
  - unknown_fraction: 1
  - silence_samples: 2000
  - sampling_strategy: natural
  - seed: 42
MODEL (variable):
  - model_type: lstm
  - dropout: 0.2
  - lstm_hidden_size: 64
  - lstm_layers: 2
  - lstm_bidirectional: True
  - transformer_d_model: 64
  - transformer_heads: 4
  - transformer_layers: 2
  - transformer_ff_dim: 128
FIT (variable):
  - epochs: 10
  - batch_size: 128
  - learning_rate: 0.001
  - weight_decay: 0.0001

Training model
Using device: cuda


Training:  30%|███       | 3/10 [03:32<08:15, 70.85s/it, loss=0.3231, lr=0.001, val_acc=0.9078, val_loss=0.3016]


KeyboardInterrupt: 

## Wczytanie zapisanych wyników

Jeśli trening został już wykonany, ta komórka pozwala wrócić do analizy bez ponownego uruchamiania modeli.

In [ ]:
summary_path = Path(OUTPUT_DIR) / "sampling_experiment_summary.csv"

if summary_path.exists():
    all_results = pd.read_csv(summary_path)

all_results.sort_values("test_accuracy", ascending=False)

## Analiza wyników

Najważniejsze porównanie to `test_accuracy` między strategiami dla tej samej architektury. Dodatkowo warto patrzeć na `epochs_trained` i `stopped_early`, bo mocne resamplowanie zmienia liczbę batchy oraz dynamikę walidacji.

In [ ]:
columns = [
    "strategy",
    "architecture",
    "train_examples",
    "unknown_train_examples",
    "test_accuracy",
    "validation_accuracy",
    "best_epoch",
    "epochs_trained",
    "stopped_early",
]

all_results[columns].sort_values(["architecture", "test_accuracy"], ascending=[True, False])

In [ ]:
accuracy_table = all_results.pivot_table(
    index="architecture",
    columns="strategy",
    values="test_accuracy",
    aggfunc="max",
).reindex(index=ARCHITECTURES, columns=SELECTED_STRATEGIES)

accuracy_table

In [ ]:
fig, axes = plt.subplots(
    1,
    len(ARCHITECTURES),
    figsize=(4.5 * len(ARCHITECTURES), 4),
    sharey=True,
)

if len(ARCHITECTURES) == 1:
    axes = [axes]

for axis, architecture in zip(axes, ARCHITECTURES):
    accuracy_table.loc[architecture, SELECTED_STRATEGIES].plot(kind="bar", ax=axis, ylim=(0, 1), rot=25)
    axis.set_title(architecture)
    axis.set_xlabel("Strategia resamplowania")
    axis.grid(axis="y", alpha=0.25)

axes[0].set_ylabel("Test accuracy")
fig.suptitle("Test accuracy dla LSTM i Transformera według strategii resamplowania")
plt.tight_layout()
plt.show()

## Krótka interpretacja do raportu

W raporcie porównaj `natural`, `unknown_undersampled` i `undersampled_then_oversampled` osobno dla LSTM i Transformera. `natural` jest punktem odniesienia bez resamplowania. Jeśli `unknown_undersampled` poprawia klasy docelowe, ale obniża wynik ogólny, oznacza to zwykle utratę informacji o zróżnicowanej klasie `unknown`. Jeśli `undersampled_then_oversampled` nie poprawia accuracy względem `natural`, ograniczenie największej klasy i duplikacja mniejszych klas prawdopodobnie nie dodają informacji, a tylko zmieniają częstość obserwowania przykładów przez model.